# Figure G00: the matched-comparator design (schematic)

**Status:** draft, Setonix (George's review of #150, comment on *Structural homology against a matched comparator*).

## Provenance

- **Panel a:** illustrative only; no real locus. It shows the design in *Candidates and the matched comparator* (Methods): a candidate is an unmatched ORF in an annotated genome with 3Di entropy ≥ 2.5 bits that overlaps no deposited CDS on either strand; its comparator partner is drawn without replacement from antisense and frameshift shadows of deposited CDS, matched on truncation status, 3Di entropy (±0.025 bits) and length (±10%), searched first in the same genome, then the same 250-genome chunk, then the whole domain.
- **Panel b:** the bacterial Swiss-Prot full-length hit rates and real-gene share from `data/figG04_matched_comparator.tsv.gz` (v7 matched comparator; the Figure 4 package).


In [ ]:
import msfig
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, FancyArrowPatch, Patch
msfig.style()
P = msfig.PALETTE
ARM = {"candidate": "#0072B2", "comparator": "#D55E00", "annotated": "#009E73", "other": "#BBBBBB"}
H = 0.5


def gene(ax, a, b, y, col, fwd=True, ls="-", tip=55):
    if fwd:
        pts = [(a, y - H/2), (b - tip, y - H/2), (b, y), (b - tip, y + H/2), (a, y + H/2)]
    else:
        pts = [(b, y - H/2), (a + tip, y - H/2), (a, y), (a + tip, y + H/2), (b, y + H/2)]
    ax.add_patch(Polygon(pts, closed=True, fc=col, ec="#222222", lw=0.6, ls=ls, zorder=3))


fig = plt.figure(figsize=(7.2, 2.7))
gs = fig.add_gridspec(1, 2, width_ratios=[1.55, 1], wspace=0.62)

# (a) design cartoon: two loci, each drawn on the forward (+) and reverse (−) strand
ax = fig.add_subplot(gs[0])
ax.axis("off")
ax.set_xlim(-200, 3420)
ax.set_ylim(-0.3, 10.6)
LOCI = {1: 8.0, 2: 2.2}
for k, y0 in LOCI.items():
    ax.plot([0, 3000], [y0, y0], color="#222222", lw=1.0, zorder=1)
    ax.text(-40, y0 + 0.7, "+", ha="right", va="center", fontsize=8)
    ax.text(-40, y0 - 0.7, "−", ha="right", va="center", fontsize=8)
ax.text(0, 10.25, "a candidate locus", ha="left", va="center", fontsize=7.5, fontweight="bold")
ax.text(0, 3.55, "partner locus", ha="left", va="center",
        fontsize=7.5, fontweight="bold")
# locus 1: annotated CDS around an unannotated gap that holds the candidate
y = LOCI[1]
gene(ax, 100, 900, y + 0.7, ARM["other"])
gene(ax, 1100, 1900, y + 0.7, ARM["annotated"])
gene(ax, 2750, 3000, y + 0.7, ARM["other"], tip=0)
gene(ax, 2050, 2650, y - 0.7, ARM["candidate"], fwd=False)
ax.text(1500, y + 1.12, "annotated CDS (positive control arm)", ha="center", va="bottom", fontsize=6.3)
ax.text(2650, y - 1.15, "candidate: unmatched ORF,\n3Di ≥ 2.5 bits, overlaps no CDS",
        ha="right", va="top", fontsize=6.3)
# locus 2: an annotated CDS whose antisense shadow is the matched partner
y = LOCI[2]
gene(ax, 100, 1300, y + 0.7, ARM["other"])
gene(ax, 1500, 3000, y + 0.7, ARM["other"])
gene(ax, 2200, 2800, y - 0.7, ARM["comparator"], fwd=False)
ax.text(2800, y - 1.15, "comparator: antisense or\nframeshift shadow of a CDS",
        ha="right", va="top", fontsize=6.3)
# the one-to-one match
ax.add_patch(FancyArrowPatch((2680, LOCI[1] - 0.7), (2830, LOCI[2] - 0.7),
                             connectionstyle="arc3,rad=-0.6", arrowstyle="<->",
                             mutation_scale=8, color="#222222", lw=0.8, ls=(0, (3, 2)), zorder=4))
ax.text(0, 5.0, "one partner per candidate, matched on\n"
        "length (±10%), 3Di (±0.025 bits), truncation",
        ha="left", va="center", fontsize=6.6, style="italic")
ax.text(-200, 10.6, "a", fontsize=10, fontweight="bold", va="top")

# (b) readout
ax = fig.add_subplot(gs[1])
m = msfig.data("figG04_matched_comparator.tsv.gz")
m = m[(m.domain == "bac") & (m.label == "v7_matched")].set_index("metric").value
c, s, a = (100 * m[f"fulllen_afdb_swissprot_{k}"] for k in ("candidate", "comparator", "annotated"))
# the numbers the caption and Results quote
assert (round(c, 1), round(s, 1), round(a, 1)) == (10.3, 7.4, 35.3)
assert abs((c - s) / (a - s) - m["share_afdb_swissprot"]) < 1e-9 and round(100 * m["share_afdb_swissprot"], 1) == 10.4
labels = ["candidates (c)", "comparator (s)", "annotated CDS (a)"]
vals = [c, s, a]
cols = [ARM["candidate"], ARM["comparator"], ARM["annotated"]]
yy = [2, 1, 0]
ax.barh(yy, vals, color=cols, height=0.6, edgecolor="#222222", lw=0.6)
for yi, v in zip(yy, vals):
    ax.text(v + 0.8, yi, f"{v:.1f}%", va="center", fontsize=7)
ax.set_yticks(yy, labels)
ax.set_xlim(0, 44)
ax.set_xlabel("full-length Foldseek hits,\nSwiss-Prot, bacteria (%)")
f = m["share_afdb_swissprot"]
ax.text(43.5, 1.55, f"real-gene share\nf = (c − s) / (a − s)\n= {f * 100:.1f}%",
        ha="right", va="center", fontsize=7,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#555555", lw=0.6))
ax.text(-0.75, 1.07, "b", transform=ax.transAxes, fontsize=10, fontweight="bold", va="top")
msfig.save(fig, "figG00_comparator_design")


## Caption (draft)

**Figure G00. How candidates are compared with a matched non-coding comparator.** **(a)** A candidate (blue) is an ORF in an annotated GTDB genome that matches no deposited CDS, has 3Di entropy of at least 2.5 bits and overlaps no CDS on either strand. Each candidate is paired with one comparator ORF (vermillion): an antisense or frameshift shadow of a deposited CDS, which is most likely non-coding, with the same truncation status, 3Di entropy within 0.025 bits and length within 10%. Partners are sought in the same genome first, then the same 250-genome chunk, then the whole domain. Length-matched deposited CDS (green) form the positive control. Coordinates are illustrative. **(b)** All three arms are searched with Foldseek, and the full-length hit rates of candidates (c), comparator (s) and annotated CDS (a) give the real-gene share f = (c − s)/(a − s): the fraction of candidates that behave like annotated genes rather than like their matched shadows. The example is bacteria against AlphaFold DB/Swiss-Prot: 10.3%, 7.4% and 35.3%, so f = 10.4%.
